# ADAS-Vision — TV3 / Milestone 3: Huấn luyện & Benchmark YOLO11 trên BDD100K

Notebook dành cho **TV3 (ML Engineer — Object Detection)** để huấn luyện, khắc phục mất cân bằng lớp và đối sánh benchmark các biến thể **YOLO11** trên **Google Colab (GPU T4)**.

* **9 lớp chuẩn (`DET_CLASSES`):** `car`, `truck`, `bus`, `motorcycle`, `bicycle`, `pedestrian`, `rider`, `traffic_light`, `traffic_sign`
* **Sản phẩm bàn giao:** `weights/detector_best.pt`, `runs/detect/.../results.csv`, bảng benchmark `outputs/detector_benchmark.md` & `.csv`

## 1. Kiểm tra GPU T4 & Môi trường

In [ ]:
!nvidia-smi

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Kết nối Google Drive, Mã nguồn & Thư viện

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Clone hoặc chuyển vào thư mục repo của dự án trên Colab
%cd /content/adas-vision

!pip install -q ultralytics pyyaml tensorboard opencv-python-headless

## 3. Kiểm tra Cấu hình Dataset (`configs/bdd_yolo.yaml`) & Phân bố Lớp

In [ ]:
from pathlib import Path

import yaml

from scripts.train_detector import count_yolo_label_instances, validate_bdd_yolo_yaml
from src.utils.constants import DET_CLASSES

# Cập nhật đường dẫn dataset BDD100K YOLO trên Colab nếu cần
data_yaml_path = Path("configs/bdd_yolo.yaml")
with data_yaml_path.open("r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

# Ví dụ: data_cfg["path"] = "/content/drive/MyDrive/datasets/bdd100k_yolo"
# with data_yaml_path.open("w", encoding="utf-8") as f:
#     yaml.safe_dump(data_cfg, f, sort_keys=False)

validate_bdd_yolo_yaml(data_yaml_path)
print("DET_CLASSES khớp hoàn toàn:", DET_CLASSES)

train_labels_dir = Path(data_cfg["path"]) / "labels" / "train"
if train_labels_dir.exists():
    counts = count_yolo_label_instances(train_labels_dir)
    for cls_name, cnt in counts.items():
        print(f"{cls_name:15s}: {cnt}")

## 4. Huấn luyện YOLO11n & YOLO11s (kèm TensorBoard & Oversampling lớp hiếm)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/detect

In [ ]:
# 4.1. Huấn luyện YOLO11n (baseline nhẹ, FPS cao)
!python scripts/train_detector.py \
    --data configs/bdd_yolo.yaml \
    --model yolo11n.pt \
    --run-name yolo11n_bdd \
    --oversample-rare \
    --output-weights weights/detector_yolo11n.pt

# 4.2. Huấn luyện YOLO11s (mô hình chính khuyến nghị)
!python scripts/train_detector.py \
    --data configs/bdd_yolo.yaml \
    --model yolo11s.pt \
    --run-name yolo11s_bdd \
    --oversample-rare \
    --output-weights weights/detector_best.pt

## 5. Benchmark So sánh Mô hình (`yolo11n` vs `yolo11s` vs `yolov8s`)

In [ ]:
!python scripts/train_detector.py \
    --benchmark-only \
    --data configs/bdd_yolo.yaml \
    --compare-models weights/detector_yolo11n.pt weights/detector_best.pt yolov8s.pt \
    --benchmark-out outputs/detector_benchmark.md

## 6. Kiểm tra Wrapper `YOLO11Detector` & Lưu Trọng số về Google Drive

In [ ]:
import shutil

import numpy as np

from src.models.detector import YOLO11Detector

detector = YOLO11Detector.from_config(
    config_path="configs/detector_yolo11.yaml",
    weights_path="weights/detector_best.pt",
)
dummy = np.zeros((720, 1280, 3), dtype=np.uint8)
print("Smoke predict ok, count:", len(detector.predict(dummy)))

drive_weights_dir = Path("/content/drive/MyDrive/adas-vision/weights")
drive_weights_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2("weights/detector_best.pt", drive_weights_dir / "detector_best.pt")
print("Đã sao lưu weights/detector_best.pt lên Google Drive.")